In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW:", DATA_RAW)

PROJECT_ROOT: c:\temp\python_learning\ml_projects\ml_projects_batch_01\05_bike_sharing_demand
DATA_RAW: c:\temp\python_learning\ml_projects\ml_projects_batch_01\05_bike_sharing_demand\data\raw


In [3]:
train = pd.read_csv(DATA_RAW / "train.csv")

print("train shape:", train.shape)
display(train.head())

train shape: (10886, 12)


,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1


In [4]:
train["datetime"] = pd.to_datetime(train["datetime"])

train["year"] = train["datetime"].dt.year
train["month"] = train["datetime"].dt.month
train["day"] = train["datetime"].dt.day
train["hour"] = train["datetime"].dt.hour
train["weekday"] = train["datetime"].dt.weekday
train["is_weekend"] = train["weekday"].isin([5, 6]).astype(int)

display(train.head())

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,year,month,day,hour,weekday,is_weekend
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16,2011,1,1,0,5,1
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40,2011,1,1,1,5,1
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32,2011,1,1,2,5,1
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13,2011,1,1,3,5,1
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1,2011,1,1,4,5,1


In [5]:
target_col = "count"
leakage_cols = ["casual", "registered"]

excluded_cols = [
    target_col,
    "casual",
    "registered",
    "datetime",
    "day",  # excluded from main feature set due to transfer risk
]

feature_cols = [col for col in train.columns if col not in excluded_cols]

print("Target:", target_col)
print("Leakage columns:", leakage_cols)
print("Excluded columns:", excluded_cols)

print("\nFeature columns:")
print(feature_cols)
print("\nNumber of features:", len(feature_cols))

Target: count
Leakage columns: ['casual', 'registered']
Excluded columns: ['count', 'casual', 'registered', 'datetime', 'day']

Feature columns:
['season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 'humidity', 'windspeed', 'year', 'month', 'hour', 'weekday', 'is_weekend']

Number of features: 13


In [6]:
X = train[feature_cols].copy()
y = train[target_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())
display(y.head())

X shape: (10886, 13)
y shape: (10886,)


,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,year,month,hour,weekday,is_weekend
0,1,0,0,1,9.84,14.395,81,0.0,2011,1,0,5,1
1,1,0,0,1,9.02,13.635,80,0.0,2011,1,1,5,1
2,1,0,0,1,9.02,13.635,80,0.0,2011,1,2,5,1
3,1,0,0,1,9.84,14.395,75,0.0,2011,1,3,5,1
4,1,0,0,1,9.84,14.395,75,0.0,2011,1,4,5,1


0    16
1    40
2    32
3    13
4     1
Name: count, dtype: int64

In [7]:
local_train_mask = train["day"] <= 15
local_valid_mask = train["day"].between(16, 19)

X_train = X.loc[local_train_mask].copy()
y_train = y.loc[local_train_mask].copy()

X_valid = X.loc[local_valid_mask].copy()
y_valid = y.loc[local_valid_mask].copy()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_valid shape:", X_valid.shape)
print("y_valid shape:", y_valid.shape)

X_train shape: (8600, 13)
y_train shape: (8600,)
X_valid shape: (2286, 13)
y_valid shape: (2286,)


In [8]:
def clipped_predictions(y_pred):
    return np.clip(y_pred, a_min=0, a_max=None)


def regression_metrics(y_true, y_pred):
    y_pred_clipped = clipped_predictions(y_pred)

    rmsle = np.sqrt(mean_squared_log_error(y_true, y_pred_clipped))
    rmse = np.sqrt(mean_squared_error(y_true, y_pred_clipped))
    mae = mean_absolute_error(y_true, y_pred_clipped)

    return {
        "RMSLE": rmsle,
        "RMSE": rmse,
        "MAE": mae,
        "min_pred_raw": np.min(y_pred),
        "min_pred_clipped": np.min(y_pred_clipped),
        "max_pred_raw": np.max(y_pred),
    }

In [9]:
frozen_params = {
    "max_iter": 200,
    "learning_rate": 0.05,
    "max_leaf_nodes": 31,
    "l2_regularization": 0.1,
    "random_state": 42,
}

frozen_model = HistGradientBoostingRegressor(**frozen_params)

print(frozen_model)

HistGradientBoostingRegressor(l2_regularization=0.1, learning_rate=0.05,
                              max_iter=200, random_state=42)


Ячейка 12 — fit on local train only, evaluate on local validation

In [10]:
frozen_model.fit(X_train, np.log1p(y_train))

y_pred_log = frozen_model.predict(X_valid)
y_pred = np.expm1(y_pred_log)

frozen_metrics = regression_metrics(y_valid, y_pred)

frozen_metrics_df = pd.DataFrame([frozen_metrics], index=["Frozen_HGB_log1p"])
frozen_metrics_df

,RMSLE,RMSE,MAE,min_pred_raw,min_pred_clipped,max_pred_raw
Frozen_HGB_log1p,0.306045,47.065693,28.274786,1.138463,1.138463,825.95043


Ячейка 13 — compare with Stage 5 reference

In [11]:
stage5_reference = {
    "RMSLE": 0.306045,
    "RMSE": 47.065693,
    "MAE": 28.274786,
}

comparison_with_stage5 = frozen_metrics_df[["RMSLE", "RMSE", "MAE"]].copy()

for metric, value in stage5_reference.items():
    comparison_with_stage5[f"Stage5_reference_{metric}"] = value
    comparison_with_stage5[f"delta_vs_stage5_{metric}"] = (
        comparison_with_stage5[metric] - value
    )

comparison_with_stage5

,RMSLE,RMSE,MAE,Stage5_reference_RMSLE,delta_vs_stage5_RMSLE,Stage5_reference_RMSE,delta_vs_stage5_RMSE,Stage5_reference_MAE,delta_vs_stage5_MAE
Frozen_HGB_log1p,0.306045,47.065693,28.274786,0.306045,-4.445287e-08,47.065693,-3.591456e-07,28.274786,-2.232781e-08


Ячейка 14 — inference requirements draft

In [12]:
inference_feature_requirements = {
    "required_raw_columns": [
        "datetime",
        "season",
        "holiday",
        "workingday",
        "weather",
        "temp",
        "atemp",
        "humidity",
        "windspeed",
    ],
    "derived_features": [
        "year",
        "month",
        "hour",
        "weekday",
        "is_weekend",
    ],
    "excluded_columns": [
        "count",
        "casual",
        "registered",
        "day",
    ],
}

inference_feature_requirements

{'required_raw_columns': ['datetime',
  'season',
  'holiday',
  'workingday',
  'weather',
  'temp',
  'atemp',
  'humidity',
  'windspeed'],
 'derived_features': ['year', 'month', 'hour', 'weekday', 'is_weekend'],
 'excluded_columns': ['count', 'casual', 'registered', 'day']}

## Stage 6 final validation summary conclusions

### Important methodological note

This is not a strict final held-out test evaluation.

Reason:

- Official Kaggle `test.csv` has no `count`.
- The local validation split days 16–19 was already used during Stage 3–5 model comparison and limited tuning.
- Therefore, this notebook provides a final validation summary for the frozen candidate using the established local time-aware validation scheme.

### Frozen candidate

Model:

- `HistGradientBoostingRegressor`

Target strategy:

- fit on `log1p(count)`
- inverse prediction transform with `expm1`

Frozen params:

```text
max_iter = 200
learning_rate = 0.05
max_leaf_nodes = 31
l2_regularization = 0.1
random_state = 42
Validation split

Same local time-aware validation split as previous stages:

local train: days 1–15
local validation: days 16–19

No random split was used.

Feature set

Main feature set:

season
holiday
workingday
weather
temp
atemp
humidity
windspeed
year
month
hour
weekday
is_weekend

Excluded from X:

count
casual
registered
raw datetime
day
Leakage checks

Passed:

official Kaggle test.csv was not used
casual not in X
registered not in X
count not in X
raw datetime not in X
day excluded from the main feature set
no Kaggle submission created
no final model saved
Why log1p target remains the selected policy

log1p(count) improved RMSLE substantially in Stage 5.

Since RMSLE is the primary metric, the frozen candidate keeps the log-target policy.

Expected inference requirements

At inference time, input data must contain:

datetime
season
holiday
workingday
weather
temp
atemp
humidity
windspeed

The inference pipeline must derive from datetime:

year
month
hour
weekday
is_weekend

The model must not require:

count
casual
registered
day
Limitation

Because this validation split was used during model selection and limited tuning, the reported metrics may be somewhat optimistic for true unseen data.

This result should be treated as a frozen candidate confirmation, not as an independent final test estimate.


---

## 4. Что должно получиться

Notebook:

```text
notebooks/05_final_validation_summary.ipynb